<a href="https://colab.research.google.com/github/Bibek-Dhakal/ml-playground-for-applied-search-intelligence/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of Analysis:**
One row in my analytical frame represents one unique content item (`content_hash_id`) for a specific client (`client_hash_id`) aggregated at the decision point.

**Time Window:**
I am analyzing a mid-panel month (March 2026). My target outcome window is `2026-03-01` to `2026-03-31`, and my features will be strictly aggregated from the prior 90 days to ensure zero forward-leakage.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

*   **Features (Max 5):**
    1. `impressions_90d` (Observable past search visibility, knowable at decision moment)
    2. `avg_position` (Observable past ranking performance, knowable at decision moment)
    3. `content_age_days` (Known metadata, knowable at decision moment)
    4. `word_count` (Known metadata, knowable at decision moment)
    5. `engagement_rate` (Observable past GA4 engagement, knowable at decision moment)
*   **Label/Proxy:** `target_decline` (A binary flag created from `trend_direction == 'down'`).
*   **Context:** `client_hash_id`, `content_hash_id`, `report_date` (Used purely for grouping and joins, never fed to the model).
*   **Excluded:** `trend_pct`, `priority_score`, `action_type`. **Why:** These are either the exact math used to calculate the label (which causes direct data leakage) or FlyRank's internal product decisions (which causes circular logic).

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [1]:
!pip install duckdb -q
import duckdb
import pandas as pd
import os
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from google.colab import userdata

# Safely get the token
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()

# Crucial: Install and load the httpfs extension
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

# Create the secret for Hugging Face
con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

# Base path without the extra /data/ folder
base_path = "hf://datasets/FlyRank/internship-warehouse"

print("--- QUERY 1: Row Count & Date Span (Mid-Panel Month: 2026-03) ---")
query_1 = f"""
SELECT MIN(report_date) as start_date, MAX(report_date) as end_date, COUNT(*) as total_rows
FROM read_parquet('{base_path}/fact_content_daily_performance/**/*.parquet')
WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
"""
display(con.execute(query_1).df())

print("\n--- QUERY 2: Data Availability (IS TRUE filter) ---")
query_2 = f"""
SELECT COUNT(*) as rows_with_gsc_data
FROM read_parquet('{base_path}/fact_content_daily_performance/**/*.parquet')
WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
AND gsc_data_available IS TRUE
"""
display(con.execute(query_2).df())

print("\n--- QUERY 3: Proving the Grain (Unique combos per day) ---")
query_3 = f"""
SELECT report_date, COUNT(content_hash_id) as total_content, COUNT(DISTINCT content_hash_id) as unique_content
FROM read_parquet('{base_path}/fact_content_daily_performance/**/*.parquet')
WHERE report_date = '2026-03-15'
GROUP BY report_date
"""
display(con.execute(query_3).df())

print("\n--- THE LEAKAGE TRAP (Experiment) ---")
# Load starter dataset locally to show the trap
file_path = "../../data/raw/content_refresh_anonymized.csv"
if not os.path.exists(file_path):
    file_path = "https://raw.githubusercontent.com/Bibek-Dhakal/applied-search-intelligence/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(file_path)
df = df[df['impressions_90d'] >= 100].copy()

features = ['impressions_90d', 'avg_position', 'content_age_days', 'word_count', 'engagement_rate']
df['target_decline'] = (df['trend_direction'] == 'down').astype(int)

# The Leak: Adding 'trend_pct' (the exact data used to derive the target)
leaked_features = features + ['trend_pct']

X_leaked = df[leaked_features].fillna(0)
y = df['target_decline']
clf_leaked = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_leaked, y)
print(f"Accuracy WITH leaked feature 'trend_pct': {accuracy_score(y, clf_leaked.predict(X_leaked)):.4f} (Dangerously perfect. The model cheated!)")

# Removing the Leak
X_clean = df[features].fillna(0)
clf_clean = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_clean, y)
print(f"Accuracy WITHOUT the leak: {accuracy_score(y, clf_clean.predict(X_clean)):.4f} (Honest, generalized score based on past observables only.)")


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


ModuleNotFoundError: No module named 'google'

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named Limitations of this Slice:**
1. **Unbalanced History & GSC-only early rows:** Not all clients had GA4 tracking set up for the entire historical window. Early rows might show `ga4_data_available IS FALSE`. If my model relies on `engagement_rate`, I must handle this gracefully so the model doesn't confuse "no tracking installed yet" with "zero traffic/engagement".
2. **Window Overlaps:** When rolling up daily facts into 90-day feature windows and 30-day target windows, I must be extremely careful to shift the target window cleanly into the future to avoid row-overlap leakage.
3. **Correlation, not Causation:** This observational data can tell us a page is declining, but it can never mathematically guarantee that a metadata refresh *caused* a recovery without a controlled experiment.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.